# Salon Experiment & Causal Analysis — Data Quality and EDA

> **Synthetic data:** These records are generated for this project; they do not represent real salons, customers, or business outcomes. This notebook is descriptive EDA and does not estimate causal effects or make a rollout recommendation.

The notebook reads all ten CSVs in `data/raw/` without changing them. It locates the project from the current working directory, so it can be run from the project root or `notebooks/`.

## 1. Setup and load the raw CSVs

The setup below locates the project root, checks that all ten expected files are present, and loads them as DataFrames. It does not write to the raw files.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "data" / "raw").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/raw. Start Jupyter from the project root or notebooks/ directory."
    )
RAW_DIR = PROJECT_ROOT / "data" / "raw"

CSV_FILES = [
    "appointments.csv",
    "clients.csv",
    "outlets.csv",
    "stylists.csv",
    "services.csv",
    "products.csv",
    "product_sales.csv",
    "experiment_assignments.csv",
    "did_outlet_policy.csv",
    "data_dictionary.csv",
]
missing_files = [name for name in CSV_FILES if not (RAW_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing required raw CSV files: {missing_files}")

DATASETS = {name: pd.read_csv(RAW_DIR / name) for name in CSV_FILES}
appointments = DATASETS["appointments.csv"].copy()
clients = DATASETS["clients.csv"].copy()
outlets = DATASETS["outlets.csv"].copy()
stylists = DATASETS["stylists.csv"].copy()
services = DATASETS["services.csv"].copy()
products = DATASETS["products.csv"].copy()
product_sales = DATASETS["product_sales.csv"].copy()
experiment_assignments = DATASETS["experiment_assignments.csv"].copy()
did_outlet_policy = DATASETS["did_outlet_policy.csv"].copy()
data_dictionary = DATASETS["data_dictionary.csv"].copy()

for column in ("appointment_date",):
    appointments[column] = pd.to_datetime(appointments[column], errors="coerce")
product_sales["sale_date"] = pd.to_datetime(product_sales["sale_date"], errors="coerce")
for column in ("signup_date", "simulated_active_until"):
    clients[column] = pd.to_datetime(clients[column], errors="coerce")
for column in ("assigned_at", "experiment_start_date", "experiment_end_date"):
    experiment_assignments[column] = pd.to_datetime(
        experiment_assignments[column], errors="coerce"
    )
did_outlet_policy["policy_start_date"] = pd.to_datetime(
    did_outlet_policy["policy_start_date"], errors="coerce"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded {len(DATASETS)} CSV files from {RAW_DIR}")
print("All observations are synthetic/simulated.")

## 2. Dataset dimensions, schemas, types, missingness, IDs, dates, and relationships

This section reports observed row/column counts, ordered column names, data types, missing values, duplicate identifier counts, and observed date ranges. Date ranges show values actually present; the dataset is not required to contain either boundary date. The integrity checks below count broken references without altering any DataFrame or CSV.

In [ ]:
EXPECTED_COLUMNS = {
    "appointments.csv": ["appointment_id", "client_id", "appointment_date", "outlet_id", "stylist_id", "service_id", "appointment_status", "appointment_revenue", "discount_amount", "service_duration_minutes", "booking_channel", "cancellation_reason"],
    "clients.csv": ["client_id", "signup_date", "retention_segment", "preferred_outlet_id", "simulated_active_until"],
    "outlets.csv": ["outlet_id", "outlet_name", "region", "price_index"],
    "stylists.csv": ["stylist_id", "outlet_id", "stylist_level", "simulated_start_date"],
    "services.csv": ["service_id", "service_name", "base_price", "base_duration_minutes"],
    "products.csv": ["product_id", "product_name", "product_category", "unit_price"],
    "product_sales.csv": ["sale_id", "appointment_id", "client_id", "outlet_id", "product_id", "sale_date", "quantity", "unit_price", "discount_amount", "product_revenue"],
    "experiment_assignments.csv": ["client_id", "assignment_group", "assigned_at", "experiment_start_date", "experiment_end_date"],
    "did_outlet_policy.csv": ["outlet_id", "region", "policy_group", "is_treated", "policy_start_date", "policy_name"],
    "data_dictionary.csv": ["file_name", "column_name", "description", "synthetic_notes"],
}

shape_report = pd.DataFrame(
    [{"file": name, "rows": len(frame), "columns": len(frame.columns)}
     for name, frame in DATASETS.items()]
).set_index("file")
display(shape_report)

schema_report = pd.DataFrame(
    [{
        "file": name,
        "expected_columns": EXPECTED_COLUMNS[name],
        "actual_columns": list(frame.columns),
        "matches_exact_order": list(frame.columns) == EXPECTED_COLUMNS[name],
    } for name, frame in DATASETS.items()]
).set_index("file")
display(schema_report)

print("Data types by file:")
for name, frame in DATASETS.items():
    print(f"\n{name}")
    display(frame.dtypes.astype(str).rename("dtype").to_frame())

missing_report = pd.concat(
    {name: frame.isna().sum() for name, frame in DATASETS.items()},
    names=["file", "column"],
).rename("missing_values").to_frame()
display(missing_report[missing_report["missing_values"].gt(0)])

primary_ids = {
    "appointments.csv": "appointment_id", "clients.csv": "client_id",
    "outlets.csv": "outlet_id", "stylists.csv": "stylist_id",
    "services.csv": "service_id", "products.csv": "product_id",
    "product_sales.csv": "sale_id", "experiment_assignments.csv": "client_id",
    "did_outlet_policy.csv": "outlet_id",
}
id_report = pd.DataFrame([
    {"file": name, "id_column": column,
     "null_or_blank": int(frame[column].isna().sum() + frame[column].astype("string").str.strip().eq("").fillna(False).sum()),
     "duplicate_rows": int(frame[column].duplicated(keep=False).sum())}
    for name, column in primary_ids.items()
    for frame in [DATASETS[name]]
]).set_index("file")
display(id_report)

date_columns = {
    "appointments.csv": ["appointment_date"],
    "clients.csv": ["signup_date", "simulated_active_until"],
    "product_sales.csv": ["sale_date"],
    "experiment_assignments.csv": ["assigned_at", "experiment_start_date", "experiment_end_date"],
    "did_outlet_policy.csv": ["policy_start_date"],
}
date_rows = []
for name, columns in date_columns.items():
    for column in columns:
        dates = pd.to_datetime(DATASETS[name][column], errors="coerce").dropna()
        date_rows.append({"file": name, "column": column,
                          "observed_min": dates.min() if not dates.empty else pd.NaT,
                          "observed_max": dates.max() if not dates.empty else pd.NaT,
                          "invalid_or_missing": int(pd.to_datetime(DATASETS[name][column], errors="coerce").isna().sum())})
date_report = pd.DataFrame(date_rows).set_index(["file", "column"])
display(date_report)

client_id_set = set(clients["client_id"].dropna())
outlet_id_set = set(outlets["outlet_id"].dropna())
stylist_outlet = stylists.drop_duplicates("stylist_id").set_index("stylist_id")["outlet_id"]
stylist_id_set = set(stylists["stylist_id"].dropna())
service_id_set = set(services["service_id"].dropna())
product_id_set = set(products["product_id"].dropna())
appointment_lookup = appointments.drop_duplicates("appointment_id").set_index("appointment_id")

appointment_stylist_outlets = appointments["stylist_id"].map(stylist_outlet)
appointment_client_present = appointments["client_id"].notna()
appointment_fk_report = pd.Series({
    "unknown non-null client IDs": int((appointment_client_present & ~appointments["client_id"].isin(client_id_set)).sum()),
    "unknown outlet IDs": int((~appointments["outlet_id"].isin(outlet_id_set)).sum()),
    "unknown stylist IDs": int((~appointments["stylist_id"].isin(stylist_id_set)).sum()),
    "stylist/outlet mismatches": int((appointment_stylist_outlets.notna() & appointment_stylist_outlets.ne(appointments["outlet_id"])).sum()),
    "unknown service IDs": int((~appointments["service_id"].isin(service_id_set)).sum()),
    "walk-in/client ID mismatches": int((appointments["booking_channel"].eq("Walk-in") != appointments["client_id"].isna()).sum()),
    "missing cancellation reason on non-completed appointments": int(appointments.loc[appointments["appointment_status"].ne("Completed"), "cancellation_reason"].isna().sum()),
    "cancellation reason present on completed appointments": int(appointments.loc[appointments["appointment_status"].eq("Completed"), "cancellation_reason"].notna().sum()),
    "clients with unknown preferred outlet": int((~clients["preferred_outlet_id"].isin(outlet_id_set)).sum()),
    "unknown non-null product-sale client IDs": int((~product_sales.loc[product_sales["client_id"].notna(), "client_id"].isin(client_id_set)).sum()),
    "unknown product-sale products": int((~product_sales["product_id"].isin(product_id_set)).sum()),
    "unknown product-sale appointments": int((~product_sales["appointment_id"].isin(appointment_lookup.index)).sum()),
    "unknown experiment assignment clients": int((~experiment_assignments["client_id"].isin(client_id_set)).sum()),
    "unknown policy outlets": int((~did_outlet_policy["outlet_id"].isin(outlet_id_set)).sum()),
}, name="invalid_rows")
display(appointment_fk_report.to_frame())

matched_sales = product_sales["appointment_id"].map(appointment_lookup["appointment_status"])
sales_lookup_report = pd.Series({
    "sales linked to non-completed appointments": int((matched_sales.notna() & matched_sales.ne("Completed")).sum()),
    "product-sale client/appointment mismatches": int((product_sales["client_id"].fillna("").astype(str) != product_sales["appointment_id"].map(appointment_lookup["client_id"]).fillna("").astype(str)).sum()),
    "product-sale outlet/appointment mismatches": int((product_sales["outlet_id"].fillna("").astype(str) != product_sales["appointment_id"].map(appointment_lookup["outlet_id"]).fillna("").astype(str)).sum()),
    "product-sale date/appointment mismatches": int((product_sales["sale_date"].dt.normalize() != product_sales["appointment_id"].map(appointment_lookup["appointment_date"]).dt.normalize()).fillna(True).sum()),
}, name="invalid_rows")
display(sales_lookup_report.to_frame())

## 3. Appointment status distribution

Counts and percentages describe the generated appointment rows. Status outcomes are synthetic and should not be interpreted as real salon rates.

In [ ]:
status_order = ["Completed", "Cancelled", "No-show"]
status_counts = appointments["appointment_status"].value_counts().reindex(status_order, fill_value=0)
status_summary = pd.DataFrame({"appointments": status_counts, "percent": status_counts / len(appointments) * 100})
display(status_summary)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=status_summary.index, y=status_summary["appointments"], color=sns.color_palette()[0], ax=ax)
ax.set(title="Synthetic appointment status distribution", xlabel="Status", ylabel="Appointment rows")
plt.tight_layout()
plt.show()

## 4. Appointment trends by day, weekday, week, and month

Daily, weekly, and monthly counts show the temporal pattern in the generated appointments. The weekday chart compares appointment counts by weekday. These are descriptive patterns from this single synthetic period.

In [ ]:
valid_appointments = appointments.dropna(subset=["appointment_date"]).copy()
valid_appointments["weekday"] = valid_appointments["appointment_date"].dt.day_name()
valid_appointments["week"] = valid_appointments["appointment_date"].dt.to_period("W-SUN").dt.start_time
valid_appointments["month"] = valid_appointments["appointment_date"].dt.to_period("M").dt.to_timestamp()
daily_counts = valid_appointments.set_index("appointment_date").resample("D")["appointment_id"].count()
weekly_counts = valid_appointments.groupby("week").size()
monthly_counts = valid_appointments.groupby("month").size()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = valid_appointments["weekday"].value_counts().reindex(weekday_order)

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes[0, 0].plot(daily_counts.index, daily_counts.rolling(7, min_periods=1).mean(), linewidth=1)
axes[0, 0].set(title="Appointments per day (7-day rolling mean)", xlabel="Date", ylabel="Rows per day")
axes[0, 1].plot(weekly_counts.index, weekly_counts.values, marker="o", markersize=3)
axes[0, 1].set(title="Appointments by week", xlabel="Week starting", ylabel="Appointment rows")
axes[1, 0].bar(monthly_counts.index.strftime("%Y-%m"), monthly_counts.values)
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].set(title="Appointments by month", xlabel="Month", ylabel="Appointment rows")
axes[1, 1].bar(weekday_counts.index, weekday_counts.values)
axes[1, 1].tick_params(axis="x", rotation=35)
axes[1, 1].set(title="Appointments by weekday", xlabel="Weekday", ylabel="Appointment rows")
plt.tight_layout()
plt.show()
monthly_counts.to_frame("appointment_rows")

## 5. Service revenue, discounts, and product revenue

Service revenue and appointment discounts come from `appointments.csv`; retail product revenue comes separately from `product_sales.csv`. The measures are reported and plotted separately to avoid conflating them. Non-completed appointment rows carry zero service revenue in this simulation.

In [ ]:
appointments["month"] = appointments["appointment_date"].dt.to_period("M").dt.to_timestamp()
product_sales["month"] = product_sales["sale_date"].dt.to_period("M").dt.to_timestamp()
service_monthly = appointments.groupby("month").agg(
    service_revenue=("appointment_revenue", "sum"),
    service_discounts=("discount_amount", "sum"),
)
product_monthly = product_sales.groupby("month")["product_revenue"].sum().rename("product_revenue")
revenue_summary = service_monthly.join(product_monthly, how="outer").fillna(0)
display(revenue_summary.round(2))

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
revenue_summary[["service_revenue", "service_discounts"]].plot(ax=axes[0], marker="o")
axes[0].set(title="Synthetic service revenue and discounts by month", ylabel="Amount (synthetic currency units)")
revenue_summary["product_revenue"].plot(ax=axes[1], color=sns.color_palette()[2], marker="o")
axes[1].set(title="Synthetic product sales revenue by month", xlabel="Month", ylabel="Product revenue (synthetic currency units)")
plt.tight_layout()
plt.show()

print("Total service revenue:", round(appointments["appointment_revenue"].sum(), 2))
print("Total service discounts:", round(appointments["discount_amount"].sum(), 2))
print("Total product revenue (separate table):", round(product_sales["product_revenue"].sum(), 2))

## 6. Client retention segments and repeat visits

Segment counts come from the synthetic client dimension. Visit counts are joined back to all clients so clients with no observed appointment rows are retained in the per-client distribution.

In [ ]:
retention_counts = clients["retention_segment"].value_counts(dropna=False).rename("clients").to_frame()
display(retention_counts)
client_visits = appointments.loc[appointments["client_id"].notna()].groupby("client_id").size().rename("appointment_rows")
client_retention = clients[["client_id", "retention_segment"]].merge(client_visits, on="client_id", how="left")
client_retention["appointment_rows"] = client_retention["appointment_rows"].fillna(0).astype(int)
visits_by_segment = client_retention.groupby("retention_segment")["appointment_rows"].agg(["count", "mean", "median", "sum"]).sort_values("mean", ascending=False)
display(visits_by_segment)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=clients, x="retention_segment", order=retention_counts.index, ax=axes[0])
axes[0].tick_params(axis="x", rotation=30)
axes[0].set(title="Synthetic client counts by retention segment", xlabel="Segment", ylabel="Clients")
sns.boxplot(data=client_retention, x="retention_segment", y="appointment_rows", order=retention_counts.index, showfliers=False, ax=axes[1])
axes[1].tick_params(axis="x", rotation=30)
axes[1].set(title="Appointment rows per client by segment", xlabel="Segment", ylabel="Appointment rows")
plt.tight_layout()
plt.show()

## 7. Outlet, region, stylist, service, and product distributions

These plots describe where generated appointments and sales are concentrated. Outlet and stylist volumes include all appointment statuses unless the label says otherwise; product distributions count product-sale lines.

In [ ]:
outlet_distribution = appointments.groupby("outlet_id").size().sort_values(ascending=False)
region_map = outlets.set_index("outlet_id")["region"]
appointment_regions = appointments["outlet_id"].map(region_map).value_counts()
stylist_distribution = appointments["stylist_id"].value_counts().head(20)
service_distribution = appointments["service_id"].map(services.set_index("service_id")["service_name"]).value_counts()
product_distribution = product_sales["product_id"].map(products.set_index("product_id")["product_name"]).value_counts()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0, 0].bar(outlet_distribution.index, outlet_distribution.values)
axes[0, 0].tick_params(axis="x", rotation=70)
axes[0, 0].set(title="Appointment rows by outlet", xlabel="Outlet", ylabel="Rows")
axes[0, 1].bar(appointment_regions.index, appointment_regions.values)
axes[0, 1].set(title="Appointment rows by region", xlabel="Region", ylabel="Rows")
axes[0, 2].barh(stylist_distribution.index[::-1], stylist_distribution.values[::-1])
axes[0, 2].set(title="Top 20 stylists by appointment rows", xlabel="Rows", ylabel="Stylist")
axes[1, 0].bar(service_distribution.index, service_distribution.values)
axes[1, 0].tick_params(axis="x", rotation=65)
axes[1, 0].set(title="Appointment rows by service", xlabel="Service", ylabel="Rows")
axes[1, 1].bar(product_distribution.index, product_distribution.values)
axes[1, 1].tick_params(axis="x", rotation=65)
axes[1, 1].set(title="Product-sale lines by product", xlabel="Product", ylabel="Sales lines")
axes[1, 2].axis("off")
plt.tight_layout()
plt.show()

print("Outlet rows:", len(outlets), "| stylist rows:", len(stylists), "| service rows:", len(services), "| product rows:", len(products))

## 8. Experiment assignment balance and pre-experiment client characteristics

Assignment is at `client_id`. These charts and tables describe baseline characteristics; they do not estimate a treatment effect. The pre-experiment window ends before 2026-09-01 and includes July–August policy-period appointments, so differences by assignment are not causal results.

**Recommended later A/B estimand:** create one row per randomized client; sum service revenue (`appointment_revenue`) during 2026-09-01 through 2026-10-31; retain every assigned client and set revenue to zero if there were no experiment-period appointments. Compare Treatment and Control at the client level and report an effect size and 95% confidence interval. Never treat repeated appointment rows as independent randomized observations.

In [ ]:
assignment_counts = experiment_assignments["assignment_group"].value_counts().reindex(["Control", "Treatment"], fill_value=0)
display(assignment_counts.rename("clients").to_frame())

pre_start = appointments["appointment_date"].min()
experiment_start = pd.Timestamp("2026-09-01")
pre_appointments = appointments.loc[
    appointments["client_id"].notna()
    & appointments["appointment_date"].lt(experiment_start)
].copy()
pre_client_metrics = pre_appointments.groupby("client_id").agg(
    pre_appointment_rows=("appointment_id", "size"),
    pre_completed_appointments=("appointment_status", lambda values: values.eq("Completed").sum()),
    pre_service_revenue=("appointment_revenue", "sum"),
)
pre_clients = experiment_assignments[["client_id", "assignment_group"]].merge(
    clients[["client_id", "signup_date", "retention_segment"]], on="client_id", how="left"
).merge(pre_client_metrics, left_on="client_id", right_index=True, how="left")
for column in ("pre_appointment_rows", "pre_completed_appointments", "pre_service_revenue"):
    pre_clients[column] = pre_clients[column].fillna(0)

pre_group_summary = pre_clients.groupby("assignment_group").agg(
    clients=("client_id", "nunique"),
    mean_signup_date_ordinal=("signup_date", lambda values: values.dropna().map(pd.Timestamp.toordinal).mean()),
    mean_pre_appointment_rows=("pre_appointment_rows", "mean"),
    mean_pre_completed_appointments=("pre_completed_appointments", "mean"),
    mean_pre_service_revenue=("pre_service_revenue", "mean"),
)
display(pre_group_summary)
display(pd.crosstab(pre_clients["retention_segment"], pre_clients["assignment_group"], normalize="columns").round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=pre_clients, x="assignment_group", y="pre_appointment_rows", errorbar=None, ax=axes[0])
axes[0].set(title="Mean pre-experiment appointment rows per client", xlabel="Randomized group", ylabel="Rows per client")
sns.barplot(data=pre_clients, x="assignment_group", y="pre_service_revenue", errorbar=None, ax=axes[1])
axes[1].set(title="Mean pre-experiment service revenue per client", xlabel="Randomized group", ylabel="Synthetic currency units per client")
plt.tight_layout()
plt.show()

print(f"Descriptive pre-period: observed appointment dates before {experiment_start.date()}; no causal interpretation.")

## 9. Descriptive Difference-in-Differences outlet trends

Policy assignment is at the outlet/region level: North and South are treated; East and West are controls. The lines show mean monthly appointment counts and service revenue per outlet in each group, including outlet-months with zero appointments. The vertical line marks the July 1, 2026 policy start. This is a descriptive visualization, not a DiD estimate.

**Period guidance:** July–August 2026 is the primary post-policy window, before the A/B experiment. September–October overlaps the client-level experiment and is used only for sensitivity analysis. There are four synthetic regions, so cluster-level inference is limited.

In [ ]:
policy_groups = did_outlet_policy[["outlet_id", "region", "policy_group", "is_treated"]].drop_duplicates("outlet_id")
did_appointments = appointments.merge(policy_groups[["outlet_id", "policy_group"]], on="outlet_id", how="left")
did_appointments["month"] = did_appointments["appointment_date"].dt.to_period("M").dt.to_timestamp()
outlet_month = did_appointments.groupby(["outlet_id", "month"]).agg(
    appointment_rows=("appointment_id", "size"),
    completed_appointments=("appointment_status", lambda values: values.eq("Completed").sum()),
    service_revenue=("appointment_revenue", "sum"),
).reset_index()
months = pd.date_range("2025-11-01", "2026-10-01", freq="MS")
complete_index = pd.MultiIndex.from_product(
    [policy_groups["outlet_id"], months], names=["outlet_id", "month"]
)
outlet_month = outlet_month.set_index(["outlet_id", "month"]).reindex(complete_index, fill_value=0).reset_index()
outlet_month = outlet_month.merge(policy_groups[["outlet_id", "policy_group"]], on="outlet_id", how="left")
group_month = outlet_month.groupby(["policy_group", "month"])[
    ["appointment_rows", "completed_appointments", "service_revenue"]
].mean().reset_index()
display(group_month.pivot(index="month", columns="policy_group", values="service_revenue").round(2))

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
sns.lineplot(data=group_month, x="month", y="service_revenue", hue="policy_group", marker="o", ax=axes[0])
axes[0].axvline(pd.Timestamp("2026-07-01"), color="black", linestyle="--", label="Policy start")
axes[0].set(title="Mean monthly service revenue per outlet (descriptive)", ylabel="Synthetic currency units / outlet")
sns.lineplot(data=group_month, x="month", y="appointment_rows", hue="policy_group", marker="o", ax=axes[1])
axes[1].axvline(pd.Timestamp("2026-07-01"), color="black", linestyle="--", label="Policy start")
axes[1].set(title="Mean monthly appointment rows per outlet (descriptive)", xlabel="Month", ylabel="Rows / outlet")
plt.tight_layout()
plt.show()
print("July-August are the primary clean post-policy months; September-October overlap the A/B period.")

## 10. Concise synthetic EDA findings and data-quality concerns

The final cell summarizes observed values from the loaded files and reports only measured data-quality issues from the checks above. Any patterns are specific to this synthetic simulation and must not be presented as real-world findings. No treatment-effect estimate or decision recommendation is produced.

In [ ]:
observed_dates = appointments["appointment_date"].dropna()
status_pct = (appointments["appointment_status"].value_counts(normalize=True) * 100).round(1)
top_service = appointments["service_id"].value_counts().head(1)
top_outlet = appointments["outlet_id"].value_counts().head(1)
print("### Observed synthetic EDA summary")
print(f"- Loaded {len(appointments):,} appointment rows and {clients['client_id'].nunique():,} unique client IDs.")
print(f"- Observed appointment date range: {observed_dates.min().date() if not observed_dates.empty else 'none'} to {observed_dates.max().date() if not observed_dates.empty else 'none'}.")
print("- Appointment status shares (%):", status_pct.to_dict())
print(f"- Total appointment service revenue: {appointments['appointment_revenue'].sum():,.2f} synthetic currency units.")
print(f"- Total product sales revenue (separate table): {product_sales['product_revenue'].sum():,.2f} synthetic currency units.")
print("- Experiment group counts:", assignment_counts.to_dict())
if not top_service.empty:
    service_name = services.set_index("service_id")["service_name"].get(top_service.index[0], top_service.index[0])
    print(f"- Most frequent service label: {service_name} ({top_service.iloc[0]:,} appointment rows).")
if not top_outlet.empty:
    print(f"- Highest appointment-row outlet: {top_outlet.index[0]} ({top_outlet.iloc[0]:,} rows).")

quality_concerns = []
if not missing_files:
    pass
else:
    quality_concerns.append(f"Missing raw files: {missing_files}")
schema_mismatches = schema_report.loc[~schema_report["matches_exact_order"]]
if not schema_mismatches.empty:
    quality_concerns.append(f"Schema mismatches in {len(schema_mismatches)} files")
expected_nullable_columns = {
    "appointments.csv": {"client_id", "cancellation_reason"},
    "product_sales.csv": {"client_id"},
}
unexpected_missing = []
for name, frame in DATASETS.items():
    allowed = expected_nullable_columns.get(name, set())
    for column, count in frame.isna().sum().items():
        if count and column not in allowed:
            unexpected_missing.append(f"{name}.{column}: {count}")
if unexpected_missing:
    quality_concerns.append("Unexpected missing values: " + "; ".join(unexpected_missing))
else:
    expected_missing = {name: {column: int(count) for column, count in frame.isna().sum().items() if count}
                        for name, frame in DATASETS.items()}
    expected_missing = {name: counts for name, counts in expected_missing.items() if counts}
    print("\nIntentional nullable fields (counts):", expected_missing)
duplicate_rows = int(id_report["duplicate_rows"].sum())
empty_identifiers = int(id_report["null_or_blank"].sum())
if duplicate_rows:
    quality_concerns.append(f"{duplicate_rows:,} rows have duplicate primary/assignment IDs")
if empty_identifiers:
    quality_concerns.append(f"{empty_identifiers:,} required identifier cells are null or blank")
nonzero_fk = appointment_fk_report[appointment_fk_report.ne(0)]
if not nonzero_fk.empty:
    quality_concerns.append("Foreign-key issues: " + nonzero_fk.to_dict().__repr__())
nonzero_sales_links = sales_lookup_report[sales_lookup_report.ne(0)]
if not nonzero_sales_links.empty:
    quality_concerns.append("Product-sale link issues: " + nonzero_sales_links.to_dict().__repr__())
if quality_concerns:
    print("\n### Data-quality concerns")
    for concern in quality_concerns:
        print("-", concern)
else:
    print("\n### Data-quality concerns")
    print("- No issues were found by the checks included in this notebook. This is not a substitute for the generator's full validation.")
print("\nAll summaries above describe synthetic data only; no causal effects were estimated.")